# Download Model Qwen2-VL-2B-Instruct

Notebook untuk cek GPU dan download model dari HuggingFace ke cache lokal.

Jalankan cell demi cell dari atas ke bawah.

## 1. Import & Cek GPU

In [1]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

print(f"Torch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("!! CUDA tidak terdeteksi. Cek instalasi torch CUDA kamu sebelum lanjut.")

d:\Codelabs\TukangKayu\Nesto.ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version : 2.9.1+cu130
CUDA available: True
GPU name      : NVIDIA GeForce RTX 4070 Laptop GPU
VRAM total    : 8.6 GB


## 2. Download Processor

Download tokenizer + image processor (ukuran kecil, cepat).

In [2]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Processor berhasil di-download")

Processor berhasil di-download


## 3. Download Model

Proses ini download ~4-5 GB bobot model (BF16). Tersimpan otomatis di cache:
`~/.cache/huggingface/hub` (Windows: `C:\Users\<username>\.cache\huggingface\hub`).

Kalau koneksi terputus, tinggal jalankan ulang cell ini — huggingface_hub otomatis resume.

In [3]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Model berhasil di-download dan dimuat ke GPU")

Loading weights: 100%|██████████| 729/729 [00:05<00:00, 126.30it/s]


Model berhasil di-download dan dimuat ke GPU


## 4. Verifikasi Model Ter-load dengan Benar

In [4]:
print(f"Model device : {model.device}")
print(f"Model dtype  : {model.dtype}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total params : {total_params / 1e9:.2f}B")

Model device : cuda:0
Model dtype  : torch.bfloat16
Total params : 2.21B


## 5. (Opsional) Test Inference Cepat

Cek model bisa jawab pertanyaan sederhana dari gambar contoh.

In [5]:
from qwen_vl_utils import process_vision_info

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"},
            {"type": "text", "text": "Deskripsikan gambar ini."},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)

generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)

print(output_text[0])

Gambar ini menunjukkan seorang wanita yang sedang bermain dengan anjing di pantai. Dia sedang menggenggam tangan anjing yang sedang mengekspresikan kebahagiaan dan kebersamaan. Mereka berdua terlihat sangat bahagia dan terlibat dalam kegiatan yang menyenangkan. Pemandangan pantai dengan pasir putih dan ombak laut menambah keindahan dan keaslian alam di foto ini.
